# Survival Analysis — S2Omics H&E Morphological Clusters

End-to-end patient-level survival pipeline built on top of the joint H&E clustering from `run_batch.py` (steps 1–4). Links morphology clusters to survival outcomes, then to tissue biology via IMC correlation.

## Pipeline overview

0. **Setup** — import the companion `survival analysis.py` module.
1. **Build cluster summary** — collect `cluster_image.pickle` outputs across samples into a per-patient cluster-frequency table.
2. **Merge cluster summary with clinical table** — fuzzy-match cluster IDs to the clinical table and write a merged CSV.
   - **2b.** *(optional)* manually exclude background / artifact clusters before downstream analysis.
3. **Kaplan–Meier curves** — grouped KM by `Broad_treatment` for PFS and OS, plus legacy OS subgroup curves (histology / stage / smoking).
4. **Clinical group comparisons** — cluster-frequency boxplots across tissue origin, stage, smoking, and related clinical variables.
5. **Cox proportional hazards models** — lasso-regularized Cox PH within each treatment subgroup × endpoint.
6. **Cox diagnostics** — Harrell's C-index, risk-stratified KM, and best-model selection per subgroup.
7. **Univariate Cox follow-up** — one-feature-at-a-time Cox fits for the lasso-selected features.
8. **All-feature univariate + forest + volcano plots** — exhaustive single-feature Cox across every cluster, with forest and volcano visualization.
9. **IMC correlation (biological interpretation)** — Spearman correlations of H&E cluster frequencies against IMC density tables, drawn as three views (signed `-log10(p)` heatmap, plain `r` heatmap, and a `r` × significance bubble plot) plus top-pair tables.

### Inputs / outputs
- `../demo/global/` — per-sample S2Omics outputs (`cluster_image.pickle`).
- `../demo/survival/` — clinical table (`survival.txt`), IMC density CSVs, and all generated summary files.

## 0. Setup

Load the companion `survival analysis.py` module (sibling of this notebook) and expose the high-level entry points used by the steps below.

In [ ]:
from pathlib import Path
import importlib.util

module_path = Path('survival analysis.py').resolve()
if not module_path.exists():
    raise FileNotFoundError(f'Cannot find script: {module_path}')

spec = importlib.util.spec_from_file_location('survival_analysis_impl', module_path)
sa = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sa)

# Expose frequently used functions
summarize_clusters_from_root = sa.summarize_clusters_from_root
merge_cluster_with_clinical = sa.merge_cluster_with_clinical
exclude_background_clusters = sa.exclude_background_clusters
run_km_analyses = sa.run_km_analyses
run_cox_models_by_treatment = sa.run_cox_models_by_treatment
run_cox_diagnostics = sa.run_cox_diagnostics
run_univariate_cox_for_lasso_selected = sa.run_univariate_cox_for_lasso_selected
run_univariate_analysis_and_forest_plots = sa.run_univariate_analysis_and_forest_plots
plot_volcano_by_treatment = sa.plot_volcano_by_treatment
run_imc_correlation_analysis = sa.run_imc_correlation_analysis
correlate_clusters_with_imc = sa.correlate_clusters_with_imc
plot_imc_correlation_heatmap = sa.plot_imc_correlation_heatmap
plot_imc_correlation_r_heatmap = sa.plot_imc_correlation_r_heatmap
plot_imc_correlation_bubbles = sa.plot_imc_correlation_bubbles
summarize_top_imc_correlations = sa.summarize_top_imc_correlations
load_imc_density_tables = sa.load_imc_density_tables

## 1. Build cluster summary

Walk every sample folder under `../demo/global/`, read its `cluster_image.pickle`, and emit a per-patient table of cluster frequencies (`freq_cluster_*`) plus tissue area / superpixel count. Writes `cluster_summary.csv` to `../demo/survival/`.

In [ ]:
# Step 1: build cluster summary from discovered S2Omics outputs

demo_root = Path('../demo/global')              # per-patient S2Omics output folders
survival_root = Path('../demo/survival')        # clinical, IMC, and summary tables
survival_root.mkdir(parents=True, exist_ok=True)

cluster_summary_path = survival_root / 'cluster_summary.csv'

df = summarize_clusters_from_root(
    root_dir=demo_root,
    output_path=cluster_summary_path,
    patch_size=16,
    pixel_size=0.5,
)
df.head()

## 2. Merge cluster summary with clinical table

Fuzzy-match the S2Omics patient IDs against the clinical table (`survival.txt`, tab-separated) and write a single merged CSV at `../demo/survival/merged_clinical_cluster_summary.csv`. The merge key is a normalized/lowercased compact patient ID; unmatched rows are reported with `_merge` and `match_score`.

In [ ]:
# Step 2: merge cluster summary with clinical table

clinical_path = survival_root / 'survival.txt'
merged_output_path = survival_root / 'merged_clinical_cluster_summary.csv'

merged_df = merge_cluster_with_clinical(
    clinical_path=clinical_path,
    cluster_summary_path=cluster_summary_path,
    output_path=merged_output_path,
)

preview_columns = [
    'patient_id',
    'cluster_patient_id',
    'clinical_patient_id',
    'OS_days',
    'OS_event',
    'PFS_days',
    'PFS_event',
    'Stage',
    'Age_dicho',
    'Sex',
    'smoking_status',
]
existing_columns = [col for col in preview_columns if col in merged_df.columns]
merged_df[existing_columns].head()

## 2b. Manually exclude background clusters (optional)

Some morphology clusters represent **background tissue, slide artifacts, white-space, or fixation marks** rather than biologically meaningful structures. Drop them here so they don't pollute the KM, Cox, univariate, and IMC steps below.

**How to identify background clusters**

1. Open a few representative `cluster_image_num_clusters_*.jpg` files from `../demo/global/<sample>/S2Omics_output/image_files/`.
2. Look for clusters that consistently fall on slide background, edges, tears, or stain artifacts.
3. ⚠️ **Off-by-one note:** the JPG legend labels clusters `1..N`, while the pickle / CSV / `freq_cluster_*` columns use `0..N-1`. Subtract 1 from the JPG legend number to get the ID for `background_cluster_ids` below. Example: "cluster 7" in the legend → `6` in the list.

Set `background_cluster_ids = []` to skip filtering. With `renormalize=True` (default) the remaining cluster frequencies are rescaled to sum to 1 per row — i.e. they become "fraction of *non-background* tissue".

In [ ]:
# Step 2b: drop background clusters from merged_df (zero-indexed pickle IDs)

# Edit this list with the cluster IDs to exclude. Empty list = no-op.
background_cluster_ids = [2, 7, 17, 18]

merged_df = exclude_background_clusters(
    merged_df,
    background_cluster_ids=background_cluster_ids,
    renormalize=True,
)

# Quick sanity check — list the cluster columns that survive into the rest of the pipeline.
sa.get_cluster_feature_columns(merged_df)

## 3. Kaplan–Meier curves by Broad_treatment and legacy OS subgroups

Grouped Kaplan–Meier analysis for both endpoints (PFS, OS) stratified by `Broad_treatment`, plus legacy OS subgroup curves (histology / stage / smoking) kept for comparison.

Broad_treatment endpoint plots:

- Progression-Free Survival (PFS)
  - Time: `PFS_days`
  - Event: `PFS_censoring` (`1 = progression event`, `0 = censored`)
- Overall Survival (OS)
  - Time: `OS_days`
  - Event: OS censoring column (`1 = death event`, `0 = censored`)

Legacy OS plots kept for comparison:

- Overall survival by Histology
- Overall survival by Stage
- Overall survival by Smoking

For each plot, a global log-rank test is reported when subgrouping is possible.

In [ ]:
# Step 3: Kaplan-Meier analyses

km_results, legacy_os_stats = run_km_analyses(merged_df, min_group_n=10)
km_results, legacy_os_stats

## 4. Clinical group comparisons

Compare cluster-frequency distributions across clinical variables (boxplots + strip overlays):

- Tissue origin
- Sample location within the Distant-metastasis subgroup
- Stage
- Smoking
- Histology

These are exploratory EDA plots sitting between the KM curves (section 3) and the Cox models (section 5); they help sanity-check whether any morphology cluster is confounded with a clinical grouping.

In [ ]:
# Step 4: clinical group comparison plots and statistics

def run_clinical_group_comparisons(merged_df):
    cluster_feature_cols = sa.get_cluster_feature_columns(merged_df)

    tissue_origin_col = sa._find_column(
        merged_df,
        ['Tissue_origin', 'tissue_origin', 'Tissue Origin', 'tissue origin'],
    )
    tissue_origin_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=tissue_origin_col,
        title='Tissue Origin',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    sample_location_col = sa._find_column(
        merged_df,
        ['Sample_location', 'sample_location', 'Sample Location', 'sample location'],
    )
    if tissue_origin_col is not None and sample_location_col is not None:
        distant_mask = merged_df[tissue_origin_col].astype(str).str.lower().str.contains('distant metastasis', na=False)
        distant_df = merged_df.loc[distant_mask].copy()
        distant_location_stats = sa.compare_cluster_features_by_group(
            df=distant_df,
            group_col=sample_location_col,
            title='Distant Metastasis Subgroup by Sample Location',
            cluster_cols=cluster_feature_cols,
            min_group_n=3,
            top_n_features=6,
        )
    else:
        print('[Distant Metastasis Subgroup by Sample Location] Skipped: Tissue_origin and/or Sample_location column not found.')
        distant_location_stats = None

    stage_col = sa._find_column(merged_df, ['Stage', 'stage', 'Clinical_stage', 'clinical_stage'])
    stage_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=stage_col,
        title='Stage',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    smoking_col = sa._find_column(
        merged_df,
        ['Smoking', 'smoking', 'smoking_status', 'Smoking_status', 'Smoking Status'],
    )
    smoking_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=smoking_col,
        title='Smoking',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    histology_col = sa._find_column(merged_df, ['Histology', 'histology'])
    histology_stats = sa.compare_cluster_features_by_group(
        df=merged_df,
        group_col=histology_col,
        title='Histology',
        cluster_cols=cluster_feature_cols,
        min_group_n=5,
        top_n_features=6,
    )

    return {
        'tissue_origin': tissue_origin_stats,
        'distant_location': distant_location_stats,
        'stage': stage_stats,
        'smoking': smoking_stats,
        'histology': histology_stats,
    }


clinical_group_stats = run_clinical_group_comparisons(merged_df)
clinical_group_stats

## 5. Cox proportional hazards models by Broad_treatment (PFS and OS)

Lasso-regularized Cox PH models fit within each `Broad_treatment` subgroup for both endpoints:

- Progression-Free Survival (PFS) — time `PFS_days`, event `PFS_censoring` (`1 = progression`, `0 = censored`)
- Overall Survival (OS) — time `OS_days`, event `OS_censoring` (`1 = death`, `0 = censored`)

For each endpoint × treatment subgroup, an elastic-net (L1) Cox model is fit; highly correlated features are dropped first and covariates are min–max normalized.

In [ ]:
# Step 5: Cox models by treatment subgroup

cox_run_registry, cox_overview_df = run_cox_models_by_treatment(
    merged_df,
    min_subgroup_n=20,
)
cox_overview_df

## 6. Cox model diagnostics and validation (by endpoint and Broad_treatment)

Validate every Cox fit from section 5 using:

- Harrell's concordance index (C-index)
- Approximate log-likelihood and sparsity summaries
- Risk-stratified Kaplan–Meier curves from predicted risk

Higher C-index means better discrimination (0.5 = random, 1.0 = perfect). The best model per endpoint × treatment subgroup is picked automatically for the KM plot.

In [ ]:
# Step 6: Cox diagnostics and risk-stratified KM

diagnostics_df, best_models = run_cox_diagnostics(
    cox_run_registry,
    max_plots=8,
)

diagnostics_df, best_models

## 7. Univariate Cox follow-up for lasso-selected features

Run one-feature-at-a-time Cox PH models for every feature with a non-zero coefficient in the lasso model, separately for each endpoint × `Broad_treatment` subgroup. Reports univariate hazard ratio, p-value, and 95% CI alongside the lasso coefficient.

In [ ]:
# Step 7: univariate Cox for lasso-selected features

univariate_lasso_df, univariate_lasso_skipped = run_univariate_cox_for_lasso_selected(
    cox_run_registry,
    coef_threshold=1e-8,
 )

univariate_lasso_df, univariate_lasso_skipped

## 8. All-feature univariate Cox, lasso comparison, forest plots & volcano plots

Four analyses in sequence:

1. **Univariate Cox for all features** — fit a one-feature-at-a-time Cox PH for every covariate in the design matrix, per endpoint × `Broad_treatment` subgroup. Reports HR, 95% CI, and p-value.
2. **Univariate vs. lasso comparison** — cross-reference the univariate results against the lasso-selected features. Categories:
   - **both** — lasso-selected AND univariately significant (p < 0.05)
   - **lasso_only** — lasso-selected but not univariately significant
   - **uni_only** — univariately significant but not lasso-selected
   - **neither**
3. **Forest plots** — one plot per endpoint × treatment subgroup showing the top features ranked by univariate p-value. Each point is the HR with a horizontal 95% CI bar; a dashed vertical line at HR = 1 separates protective (HR < 1) from risky (HR > 1). Points are color-coded by lasso / univariate agreement.
4. **Volcano plots** — one plot per endpoint × treatment subgroup showing every feature: X = `log2(HR)` (protective ← 0 → risky), Y = `-log10(p)`. Dashed lines mark HR = 1 and p = 0.05. Red = risky & significant, blue = protective & significant, grey = ns. Lasso-selected features are outlined in black. Top-N features by p-value are text-labeled.

**Cell 8b** below re-plots only the volcanoes with a customizable `p_threshold` / `label_top_n` — useful for exploring different significance thresholds without re-running the univariate fits.

In [ ]:
# Step 8: all-feature univariate Cox, lasso comparison, and forest plots

all_uni_df, comparison_df = run_univariate_analysis_and_forest_plots(
    cox_run_registry,
    coef_threshold=1e-8,
    p_threshold=0.05,
    max_features=30,
)

all_uni_df, comparison_df

In [ ]:
# Step 8b: redraw volcano plots with custom thresholds / label count

plot_volcano_by_treatment(
    all_uni_df,
    cox_run_registry,
    coef_threshold=1e-8,
    p_threshold=0.05,
    label_top_n=12,
)

## 9. Biological interpretation of clusters — IMC correlation analysis

Correlates H&E cluster frequencies (`freq_cluster_*`) against four IMC density tables in `../demo/survival/` to give the morphology clusters a cellular interpretation:

- `broad_clusters_expr_density_p1.csv` — cellular-neighborhood density (15 nearest neighbors)
- `local_clusters_expr_density_p1.csv` — local cellular-neighborhood density (5 nearest neighbors)
- `celltypes_density_dat_p1.csv` — cell-type density
- `clustered_cells_density_dat_p1.csv` — clustered cell-type density

For each IMC table: inner-join on `clinical_patient_id` ↔ `immucan_id`, compute **column-wise Spearman correlations** between every H&E cluster frequency and every IMC feature, and apply **Benjamini–Hochberg FDR correction** per table.

Three complementary views are drawn for every IMC table:

1. **Signed `-log10(p)` heatmap** — color intensity encodes significance, sign encodes correlation direction. Best for spotting "highly significant" cells at a glance.
2. **Spearman `r` heatmap** — color in `[-1, 1]` directly shows correlation **direction and strength** without conflating significance. Cells with FDR < threshold are still marked with `*`.
3. **Bubble plot** — color = Spearman `r` (direction & strength), dot size = `-log10(p)` (significance). Combines both axes of information in a single panel.

A **top-pairs table** is produced per IMC file across all H&E clusters, sorted by FDR — strongest biological associations surface regardless of which clusters were flagged as protective or risky.

In [ ]:
# Step 9: IMC correlation analysis (biological interpretation of clusters)

imc_corr_df, imc_top_df = run_imc_correlation_analysis(
    merged_df=merged_df,
    imc_dir=survival_root,
    method='spearman',
    p_threshold=0.05,
    use_fdr=True,
    top_n=20,
    min_n=20,
)

imc_corr_df.head(), imc_top_df.head()